In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)

In [2]:

np.random.seed(9)


In [3]:
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [4]:
df= df.iloc[:,0:-1]
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [5]:
df.iloc[1,0] = np.NaN
df.iloc[3,1] = np.NaN
df.iloc[-1,-1] = np.NaN
df.head()

C:\Users\BIT\AppData\Local\Temp\ipykernel_23060\4209224845.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1,0] = np.NaN
C:\Users\BIT\AppData\Local\Temp\ipykernel_23060\4209224845.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3,1] = np.NaN
C:\Users\BIT\AppData\Local\Temp\ipykernel_23060\4209224845.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1,-1] = np.NaN


,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


In [6]:
#standard code just change the column name
from sklearn.linear_model import LinearRegression
import pandas as pd

# 1. Initial Mean Imputation (Step 0)
df_current = df.copy()
for col in ['R&D Spend', 'Administration', 'Marketing Spend']:
    df_current[col] = df[col].fillna(df[col].mean())

# --- SETTINGS ---
total_iterations = 18 # Change this to 2, 5, or 10 as needed
features = ['R&D Spend', 'Administration', 'Marketing Spend']
df_old = df_current.copy() # To keep track of the very first mean-imputed state

# 2. Outer Loop for Iterations
for i in range(1, total_iterations + 1):
    # Store the state before this iteration starts to see the change
    df_before_iter = df_current.copy() 
    
    # 3. Inner Loop (MICE logic)
    for target in features:
        # Remove only the values that were originally missing
        df_current.loc[df[target].isnull(), target] = None 
        
        # Split data for training
        train_df = df_current[df[target].notnull()]
        predict_df = df_current[df[target].isnull()]
        
        cols_to_use = [c for c in features if c != target]
        
        # Fit Linear Regression
        lr = LinearRegression()
        lr.fit(train_df[cols_to_use], train_df[target])
        
        # Fill the missing values with predictions
        df_current.loc[df[target].isnull(), target] = lr.predict(predict_df[cols_to_use])

    # 4. Print Results for this specific iteration
    print(f"\n--- Result after Iteration {i} ---")
    diff = df_current - df_before_iter
    print(diff)


--- Result after Iteration 1 ---
    R&D Spend  Administration  Marketing Spend
21   0.000000        0.000000         0.000000
37  13.891587        0.000000         0.000000
2    0.000000        0.000000         0.000000
14   0.000000       -0.186382         0.000000
44   0.000000        0.000000         2.351847

--- Result after Iteration 2 ---
    R&D Spend  Administration  Marketing Spend
21   0.000000        0.000000         0.000000
37   0.687117        0.000000         0.000000
2    0.000000        0.000000         0.000000
14   0.000000        0.167119         0.000000
44   0.000000        0.000000         7.788589

--- Result after Iteration 3 ---
    R&D Spend  Administration  Marketing Spend
21    0.00000         0.00000          0.00000
37    3.05908         0.00000          0.00000
2     0.00000         0.00000          0.00000
14    0.00000         1.04506          0.00000
44    0.00000         0.00000         23.94892

--- Result after Iteration 4 ---
    R&D Spend  Adm

In [7]:
print("Final Imputed DataFrame:")
print(df_current)

Final Imputed DataFrame:
    R&D Spend  Administration  Marketing Spend
21   8.000000       15.000000        30.000000
37  26.718364        5.000000        20.000000
2   15.000000       10.000000        41.000000
14  12.000000       13.022368        26.000000
44   2.000000       15.000000        70.692067
